In [ ]:
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder

# Carregar as bases de treino e teste
train = pd.read_csv('treino_out_deliverytime.csv')
test = pd.read_csv('Teste_Final.csv')

# Definir a variável target
y_train = train['delivery_time (days)']

# Remover a coluna 'delivery_time (days)' do conjunto de treino e 'order_id' de ambos os conjuntos
#X_intermediate = train.drop(columns=['approval_to_carrier_days', 'carrier_to_customer_days', 'delivery_time (days)', 'payment_value', 'seller_city', 'seller_state', 'product_category_name', 'product_height_cm','purchase_to_approval_days'])
X_train = train.drop(columns=['order_id', 'delivery_time (days)'])
X_test = test.drop(columns=['order_id'])

# Identificar variáveis categóricas e numéricas
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remover colunas indesejadas das variáveis numéricas
numerical_cols = [col for col in numerical_cols if col not in ['approval_to_carrier_days', 'carrier_to_customer_days']]

# Preencher valores ausentes nas colunas numéricas selecionadas em X_test com a média das colunas de X_train
X_test[numerical_cols] = X_test[numerical_cols].fillna(X_train[numerical_cols].mean())

# Criar e aplicar LabelEncoders para cada coluna categórica
label_encoders = {}
for col in categorical_cols:
    X_train[col] = X_train[col].fillna(X_train[col].mode().iloc[0])
    X_test[col] = X_test[col].fillna(X_train[col].mode().iloc[0])
    
    # Criar e ajustar o LabelEncoder
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    
    # Transformar os dados de teste com o LabelEncoder, atribuindo -1 para categorias não vistas
    X_test[col] = X_test[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
    label_encoders[col] = le

# Alinhar X_train e X_test com as colunas definidas, excluindo as colunas indesejadas
columns_to_use = [col for col in X_train.columns if col in numerical_cols + categorical_cols]
X_train = X_train[columns_to_use]
X_test = X_test[columns_to_use]

# Instanciar o modelo CatBoost
model = CatBoostRegressor(
    iterations=1300,
    learning_rate=0.05,
    depth=6,
    silent=True,
    cat_features=categorical_cols  # Passa as variáveis categóricas
)

# Treinar o modelo
model.fit(X_train, y_train)

# Fazer previsões na base de teste
predictions = model.predict(X_test)

# Criar o DataFrame final com 'order_id' e as previsões
output = pd.DataFrame({
    'order_id': test['order_id'],
    'delivery_time (days)': predictions
})

# Salvar o resultado em um arquivo CSV
output.to_csv('predicoes_delivery_time.csv', index=False)

print("Arquivo 'predicoes_delivery_time.csv' salvo com sucesso!")

